# Cod generation for LSA pipelines

Anton Antonov   
May 2026

----

## Setup

Load packages:

In [19]:
use ML::LatentSemanticAnalyzer;
use DSL::English::LatentSemanticAnalysisWorkflows;
use DSL::Examples;
use LLM::Resources;
use LLM::Resources::DSLTranslation;
use ML::NLPTemplateEngine;

LLM access configuration:

In [20]:
sink my $llm-evaluator = llm-evaluator(llm-configuration('Ollama', model => 'gemma3:4b'));

Texts to do verifications with: 

In [21]:
# Collection of texts
my @dsAbstracts = ML::LatentSemanticAnalyzer::Utilities::get-abstracts-dataset();

# Remove non-strings
@dsAbstracts .= grep({ $_<Abstract> ~~ Str:D });

say "@dsAbstracts.elems : {@dsAbstracts.elems}";
my %abstracts = @dsAbstracts.map(*<ID>) Z=> @dsAbstracts.map(*<Abstract>);

srand(12);
%abstracts = %abstracts.pick(200);
say "\%abstracts.elems : {%abstracts.elems}";

@dsAbstracts.elems : 581
%abstracts.elems : 200


----

## Code generation

### Workflow spec

Natural language spec (to be translated to Raku code):

In [22]:
sink my $spec = q:to/END/;
create from textual data %abstracts;
create document term matrix with without stemming;
show document term matrix statistics;
apply term weight functions IDF, None, Cosine;
extract 60 topics with the method SVD;
echo topics table;
show statistical thesaurus for the words: function, notebook, regex using 12 synonyms per word;
show pipeline value;
echo context;
assign object to $lsaObj
END

### By grammar-based DSL translation

In [23]:
ToLatentSemanticAnalysisWorkflowCode($spec, target => 'Raku::LSAMon', format => 'code');

ML::LatentSemanticAnalyzer.new(%abstracts)
.make-document-term-matrix( stemming-rules => False)
.echo-document-term-matrix-statistics()
.apply-term-weight-functions(global-weight-func => "IDF", local-weight-func => "None", normalizer-func => "Cosine")
.extract-topics(number-of-topics => 60, method => "SVD")
.echo-topics-table( )
.echo-statistical-thesaurus(terms => ["function", "notebook", "regex"], number-of-nearest-neighbors => 12)
.echo-value()
.take-context()
==> my $lsaObj

### By DSL examples

LLM-translation by examples:

In [24]:
llm-dsl-translation($spec, :$llm-evaluator)

create-document-term-matrix(:%abstracts)
.make-document-term-matrix(:!stemming-rules, :stopWords)
.show-document-term-matrix-statistics()
.apply-term-weight-functions(global-weight-func=>'IDF', local-weight-func=>'None', normalizer-func=>'Cosine')
.extract-topics(:60number-of-topics, method=>'SVD')
.echo-topics-interpretation(:10number-of-terms, :wide-form)
.echo-statistical-thesaurus(terms => ['function','notebook','regex'].map(*.&porter), :wide-form, :12number-of-nearest-neighbors)
.show-pipeline-value()
.echo-context(:wide-form)
.ML::LatentSemanticAnalyzer.new($lsaObj)

### By specialized LLM graph

LLM-graph translation: grammar-based translation is applied first, if it fails LLM-translation is used:

In [25]:
my $gBestCode = llm-resource-graph('code-generation-by-fallback', input => {:$spec, lang => 'Raku', workflow-name => 'LSAMon', :split}, :$llm-evaluator);

LLM::Graph(size => 4, nodes => code, dsl-grammar, llm-examples, workflow-name)

Show the graph:

In [26]:
#% html
$gBestCode.dot(:svg, vertex-width => 1.2, theme => 'ortho')

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.1.1 (20251213.1925)
 -->
<!-- Pages: 1 -->
 
 
 
<!-- code -->
 
 code 
 
 code 
 
<!-- dsl-grammar -->
 
 dsl-grammar 
 
 dsl-grammar 
 
<!-- dsl-grammar->code -->
 
 dsl-grammar->code 
 
 
 
<!-- llm-examples -->
 
 llm-examples 
 
 llm-examples 
 
<!-- dsl-grammar->llm-examples -->
 
 dsl-grammar->llm-examples 
 
 
 
<!-- lang -->
 
 lang 
 
 lang 
 
<!-- lang->dsl-grammar -->
 
 lang->dsl-grammar 
 
 
 
<!-- lang->llm-examples -->
 
 lang->llm-examples 
 
 
 
<!-- llm-examples->code -->
 
 llm-examples->code 
 
 
 
<!-- spec -->
 
 spec 
 
 spec 
 
<!-- spec->dsl-grammar -->
 
 spec->dsl-grammar 
 
 
 
<!-- spec->llm-examples -->
 
 spec->llm-examples 
 
 
 
<!-- split -->
 
 split 
 
 split 
 
<!-- split->llm-examples -->
 
 split->llm-examples 
 
 
 
<!-- workflow-name -->
 
 workflow-name 
 
 workflow-name 
 
<!-- workflow-name->llm-examples -->
 
 workflow-name->llm-examples

Show the translation result:

In [27]:
$gBestCode.nodes<code><result>

ML::LatentSemanticAnalyzer.new(%abstracts)
.make-document-term-matrix( stemming-rules => False)
.echo-document-term-matrix-statistics()
.apply-term-weight-functions(global-weight-func => "IDF", local-weight-func => "None", normalizer-func => "Cosine")
.extract-topics(number-of-topics => 60, method => "SVD")
.echo-topics-table( )
.echo-statistical-thesaurus(number-of-nearest-neighbors => 12, terms => ["function", "notebook", "regex"])
.echo-value()
.take-context()
==> my $lsaObj

### By NLP template engine

In [30]:
'create from %absracts; apply LSI functions IDF, None, Cosine; extract 20 topics; show topics table; thesaurus for notebook and function.'
==> concretize(lang => "Raku", :$llm-evaluator)

my $lsaObj = LatentSemanticAnalyzer.new
.make-document-term-matrix(docs=>%absracts,
                           stop-words=>Automatic,
                           stemming-rules=>Automatic,
                           min-length=>3)
.apply-term-weight-functions(global-weight-func=>"IDF",
						   local-weight-func=>"None",
						   normalizer-func=>"Cosine")
.extract-topics(number-of-topics=>20, min-number-of-documents-per-term=>20, method=>"LSI", max-steps=>16)
.echo-topics-interpretation(number-of-terms=>20, wide-form=>True)
.echo-statistical-thesaurus(terms=>["notebook", "function"],
						  wide-form=>True,
						  number-of-nearest-neighbors=>12,
						  method=>"cosine",
						  echo-function=>&put)

----

## Verification

Grammar-based DSL translation output:

In [ ]:
ML::LatentSemanticAnalyzer.new(%abstracts)
.make-document-term-matrix( stemming-rules => False)
.echo-document-term-matrix-statistics()
.apply-term-weight-functions(global-weight-func => "IDF", local-weight-func => "None", normalizer-func => "Cosine")
.extract-topics(number-of-topics => 60, method => "SVD")
.echo-topics-table( )
.echo-statistical-thesaurus(terms => ["function", "notebook", "regex"], number-of-nearest-neighbors => 12)
.echo-value()
.echo-context()
==> my $lsaObj

NLP engine output:

In [ ]:
my $lsaObj = ML::LatentSemanticAnalyzer.new
.make-document-term-matrix(docs=>%abstracts,
                           stop-words=>Whatever,
                           stemming-rules=>Whatever,
                           min-length=>3)
.apply-term-weight-functions(global-weight-func=>"IDF",
						   local-weight-func=>"None",
						   normalizer-func=>"Cosine")
.extract-topics(number-of-topics=>20, min-number-of-documents-per-term=>20, method=>"SVD", max-steps=>16)
.echo-topics-interpretation(number-of-terms=>20, wide-form=>True)
.echo-statistical-thesaurus(terms=>["thesaurus"],
						  wide-form=>True,
						  number-of-nearest-neighbors=>12,
						  method=>"cosine",
						  echo-function=>&put)